# 01b. Tiền xử lý ảnh: OCR và mô tả ảnh bằng VLM

Chuyển thông tin trong ảnh thành text để PhoBERT dùng được:
- **OCR (PaddleOCR phát hiện vùng chữ + VietOCR nhận dạng)**: chữ xuất hiện trong ảnh (meme, ảnh chụp bình luận, biển báo...).
- **VLM (Vintern-1B)**: mô tả ngắn nội dung ảnh bằng tiếng Việt (ai/cái gì, hành động, bối cảnh, biểu cảm).

Chạy **một lần** cho toàn bộ train/public test/private test. Kết quả lưu JSON trong `paths.cache_dir`
(`ocr_v2.json`, `vlm_description_v1.json`). Bị ngắt thì chạy lại, ảnh đã xử lý sẽ được bỏ qua. Ảnh lỗi không được ghi
vào cache: script báo số ảnh còn thiếu và thoát với mã lỗi, chạy lại cell đó để thử lại.

Trên Kaggle (*Internet* bật, *GPU T4 x2*, attach `hhhoang/vimmsd-dataset`):
- OCR và mô tả VLM đều mất vài giờ cho ~13.700 ảnh, phiên tương tác dễ bị ngắt. Chạy bằng **Save Version → Save & Run All**
  (chạy nền tối đa 12 giờ, output trong `/kaggle/working` được giữ lại).
- Chạy **OCR và VLM ở hai version riêng**: đặt `RUN_OCR` / `RUN_VLM` ở cell tham số bên dưới.
- Có 2 GPU thì mỗi GPU chạy một nửa số ảnh (`NUM_GPUS = 2`), xong gộp lại thành một file.
- Bị ngắt giữa chừng: attach output của version trước làm input, đặt `RESUME_DIR` trỏ tới thư mục `cache` trong đó,
  chạy lại; ảnh đã xử lý được bỏ qua.

Xong, attach output vào các notebook train và đặt `data.image_text.dir` trỏ tới thư mục `cache` đó
(hoặc `IMAGE_TEXT_DIR` trong notebook 07).

Prompt mô tả **không nhắc tới mỉa mai**, để VLM không đưa phán đoán nhãn vào input của model.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    os.chdir("/kaggle/working" if os.path.isdir("/kaggle/working") else "/")  # chạy lại cell: không đứng trong CODE_DIR khi xoá nó
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import matplotlib.pyplot as plt
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

In [ ]:
# paddlepaddle-gpu 3.x không có trên PyPI: cài từ kho của PaddlePaddle (bản build cho CUDA 12.6).
# pip sẽ cảnh báo lệch phiên bản nvidia-*-cu12 với torch: bỏ qua nếu cell kiểm tra bên dưới chạy được.
!pip install -q paddlepaddle-gpu -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q "paddleocr>=3.0" timm einops gdown
# vietocr ghim pillow/einops/gdown bản cũ: bỏ qua dependency để không hạ phiên bản của môi trường
!pip install -q --no-deps vietocr

In [ ]:
# ===== Tham số chạy =====
RUN_OCR = True        # version 1: OCR
RUN_VLM = False       # version 2: mô tả ảnh bằng VLM (chạy riêng, mất nhiều giờ)
PREVIEW = True        # xem thử 8 ảnh trước khi chạy toàn bộ
OCR_MODEL = "vgg_seq2seq"   # vgg_seq2seq (nhanh) | vgg_transformer (chậm hơn, chính xác hơn một chút)
LIMIT = None          # vd 50: chỉ chạy 50 ảnh đầu mỗi split để đo tốc độ
RESUME_DIR = None     # vd "/kaggle/input/<output-version-trước>/cache": nối tiếp phần đã chạy
VLM_BACKEND = "vintern"  # vintern (Vintern-1B, tiếng Việt) | hf (Qwen2.5-VL-3B, dùng khi vintern lỗi)

import shutil, subprocess, torch
NUM_GPUS = max(1, torch.cuda.device_count())
print("GPU:", NUM_GPUS)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

import paddle
paddle.utils.run_check()  # phải báo PaddlePaddle chạy được trên GPU

In [ ]:
import gc
import pandas as pd
import torch
from PIL import Image
from vimmsd.data.dataset import load_all_records
from vimmsd.data.image_text import DESCRIBE_PROMPT, OCRExtractor, VLMDescriber, image_key

cfg = load_config("configs/base.yaml")
labels = list(cfg.data.labels)
records = load_all_records(cfg)
# 2 mẫu mỗi nhãn để xem thử chất lượng trước khi chạy toàn bộ
preview = [r for lab in range(len(labels)) for r in [x for x in records["train"] if x["label"] == lab][:2]]
print("prompt mô tả:", DESCRIBE_PROMPT)

cache_dir = Path(cfg.data.image_text.get("dir") or cfg.paths.cache_dir)
cache_dir.mkdir(parents=True, exist_ok=True)
if RESUME_DIR:  # chép cache của version trước (input chỉ đọc) sang thư mục ghi được
    for f in Path(RESUME_DIR).glob("*.json"):
        if not (cache_dir / f.name).exists():
            shutil.copy(f, cache_dir / f.name)
print("cache:", cache_dir, sorted(p.name for p in cache_dir.glob("*.json")))


def extract(task, extra=()):
    """Chạy scripts/extract_image_text.py, mỗi GPU một tiến trình (một phần ảnh), rồi gộp các phần lại.
    Log từng tiến trình ghi ra cache_dir/<task>_gpu<k>.log; tiến độ in ra mỗi 2 phút."""
    import time
    from vimmsd.data.image_text import merge_image_text_caches, shard_cache_name
    name = cfg.data.image_text.ocr_cache if task == "ocr" else cfg.data.image_text.description_cache
    base = [sys.executable, "scripts/extract_image_text.py", "--config", "configs/base.yaml", "--task", task, *extra]
    if LIMIT:
        base += ["--limit", str(LIMIT)]
    procs = []
    for k in range(NUM_GPUS):
        cmd = base + (["--num-shards", str(NUM_GPUS), "--shard", str(k)] if NUM_GPUS > 1 else [])
        log = open(cache_dir / f"{task}_gpu{k}.log", "w")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(k)}
        procs.append((subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env), log))
    while any(p.poll() is None for p, _ in procs):
        time.sleep(120)
        for k in range(NUM_GPUS):  # dòng tqdm cuối cùng của mỗi log
            tail = (cache_dir / f"{task}_gpu{k}.log").read_text(errors="ignore").replace("\r", "\n").strip().splitlines()
            print(f"[gpu{k}]", tail[-1][-120:] if tail else "")
    if NUM_GPUS > 1:
        shards = [cache_dir / shard_cache_name(name, k, NUM_GPUS) for k in range(NUM_GPUS)]
        merged = merge_image_text_caches(cache_dir / name, [s for s in shards if s.exists()])
        print(f"đã gộp {len(merged)} ảnh vào {cache_dir / name}")
    for k, (p, log) in enumerate(procs):
        log.close()
        print(f"[gpu{k}] mã thoát {p.returncode}")
        if p.returncode != 0:  # in phần cuối log để thấy lỗi thật, thay vì chỉ thấy cache trống
            lines = (cache_dir / f"{task}_gpu{k}.log").read_text(errors="ignore").replace("\r", "\n").strip().splitlines()
            print("\n".join(l for l in lines[-25:] if "it/s]" not in l and "s/it]" not in l))

## 1. OCR

Xem thử trên vài ảnh, sau đó chạy toàn bộ.

In [ ]:
if RUN_OCR and PREVIEW:
    ocr = OCRExtractor(rec_model=OCR_MODEL)
    for r, text in zip(preview, ocr.extract_many([r["image_path"] for r in preview])):
        r["ocr"] = text
        print(f"[{labels[r['label']]}] {text[:150]!r}")
    del ocr; gc.collect(); torch.cuda.empty_cache()

In [ ]:
if RUN_OCR:
    extract("ocr", ["--ocr-model", OCR_MODEL])

## 2. Mô tả ảnh bằng VLM

Chạy ở version riêng (`RUN_OCR = False`, `RUN_VLM = True`). Vintern xử lý từng ảnh một nên chậm hơn OCR nhiều;
với 2 GPU mỗi GPU chạy một nửa. Bị ngắt thì attach output và đặt `RESUME_DIR` để chạy tiếp.

In [ ]:
if RUN_VLM and PREVIEW:
    import time
    vlm = VLMDescriber(backend=VLM_BACKEND)
    t0 = time.time(); first = vlm(preview[0]["image_path"])  # lỗi model/thư viện sẽ hiện ngay ở đây
    print(f"{time.time() - t0:.1f}s/ảnh:", repr(first))
    fig, axes = plt.subplots(2, 4, figsize=(18, 10))
    for ax, r in zip(axes.flat, preview):
        r["description"] = vlm(r["image_path"])
        ax.imshow(Image.open(r["image_path"]).convert("RGB")); ax.axis("off")
        ax.set_title(f"[{labels[r['label']]}]\n{r['description'][:110]}", fontsize=8, wrap=True)
    plt.tight_layout(); plt.show()
    del vlm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
if RUN_VLM:
    extract("description", ["--vlm", VLM_BACKEND])

## 3. Kiểm tra kết quả

In [ ]:
def read_cache(name):  # version chỉ chạy OCR (hoặc chỉ VLM) thì file còn lại chưa có
    path = cache_dir / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

ocr_cache = read_cache(cfg.data.image_text.ocr_cache)
desc_cache = read_cache(cfg.data.image_text.description_cache)

rows = []
for split, recs in records.items():
    for r in recs:
        p = Path(r["image_path"])
        rows.append({"split": split, "id": r["id"], "image_key": image_key(p.parent, p.name),
                     "label": labels[r["label"]] if r["label"] >= 0 else "(không nhãn)",
                     "caption": r["caption"], "ocr": ocr_cache.get(image_key(p.parent, p.name)),
                     "description": desc_cache.get(image_key(p.parent, p.name))})
df = pd.DataFrame(rows)
print("thiếu trong cache OCR:", df.ocr.isna().sum(), "| thiếu trong cache mô tả:", df.description.isna().sum())
df = df.fillna("")
df["có chữ (OCR)"] = df.ocr.str.len() > 0
df["số từ OCR"] = df.ocr.str.split().str.len()
df["số từ mô tả"] = df.description.str.split().str.len()
df["mô tả rỗng"] = df.description.str.len() == 0
df.groupby("label")[["có chữ (OCR)", "số từ OCR", "số từ mô tả", "mô tả rỗng"]].mean().round(2)

In [ ]:
# mô tả giống hệt nhau lặp lại nhiều lần thường là dấu hiệu VLM trả lời chung chung
display(df.description.value_counts().head(10))
df[df.split == "train"].sample(10, random_state=0)[["label", "caption", "ocr", "description"]]

## 4. Lưu kết quả cho các bước train

Các file cache nằm trong `cache_dir` (`/kaggle/working/cache` trên Kaggle). Cell dưới chép chúng cùng một bảng tổng hợp
vào `/kaggle/working/image_text/`:
- `ocr_v2.json`, `vlm_description_v1.json`: notebook train đọc trực tiếp, qua `data.image_text.dir`.
- `image_text_v2.csv`: mỗi mẫu một dòng (split, id, image_key, nhãn, caption, OCR, mô tả), để xem và phân tích.

File trong `/kaggle/working` chỉ được giữ khi **Save Version**, hoặc tải về từ tab *Output* trước khi phiên kết thúc.

In [ ]:
export_dir = Path("/kaggle/working/image_text") if Path("/kaggle/working").exists() else Path(cfg.paths.output_dir) / "image_text"
export_dir.mkdir(parents=True, exist_ok=True)
for name in (cfg.data.image_text.ocr_cache, cfg.data.image_text.description_cache):
    if (cache_dir / name).exists():
        shutil.copy(cache_dir / name, export_dir / name)
df[["split", "id", "image_key", "label", "caption", "ocr", "description"]].to_csv(
    export_dir / "image_text_v2.csv", index=False, encoding="utf-8-sig")
shutil.make_archive(str(export_dir), "zip", export_dir)  # một file zip để tải về nhanh từ tab Output
print(export_dir, sorted(f.name for f in export_dir.iterdir()))